# Entrega 5 — Obtenção, limpeza e tratamento dos dados
**TPAE II (PSAE00320) · Equipe B** · Tema: perfil e inserção dos economistas no mercado de trabalho da Paraíba (demandante: CORECON-PB).

## Seção 1 — Objetivo, fontes e produtos
**Pergunta:** quem são e onde trabalham os economistas com vínculo formal na Paraíba?

| Fonte | Papel | Arquivo em `data/raw/` |
|---|---|---|
| RAIS 2024 (MTE) | estoque de vínculos (recorte: economistas, UF 25) | `rais_economistas_pb_2024.csv` |
| Novo CAGED 2024 (MTE) | fluxo mensal de admissões e desligamentos (recorte) | `caged_economistas_pb_2024.csv` |
| IBGE — DTB | códigos e nomes dos municípios da PB | `RELATORIO_DTB_BRASIL_MUNICIPIO.xls` |
| INEP — Censo da Educação Superior 2024 | oferta de cursos (recorte, opcional neste notebook) | `inep_economia_pb_2024.csv` |

**Regra:** RAIS (estoque) e CAGED (fluxo) ficam em tabelas separadas e **não são somadas**.

**Produtos:** `data/processed/rais_economistas_pb_2024_tratada.csv`, `data/processed/caged_economistas_pb_2024_tratada.csv`, `data/processed/municipios_pb.csv`, tabelas de conferência em `outputs/`.

## Seção 2 — Bibliotecas e configuração reproduzível

In [1]:
import re, unicodedata, platform
from pathlib import Path
import numpy as np
import pandas as pd

print('Python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)

# Caminhos relativos à raiz do repositório (o notebook fica em notebooks/)
RAIZ = Path('..') if Path('../data').exists() else Path('.')
RAW, PROC, OUT = RAIZ/'data/raw', RAIZ/'data/processed', RAIZ/'outputs'
for p in (PROC, OUT): p.mkdir(parents=True, exist_ok=True)

ANO = 2024
CBO_VALIDOS = {'251205','251210','251215','251220','251225','251230','251235'}  # família 2512
conferencia = []   # (etapa, base, linhas_antes, linhas_depois, observacao)
def registrar(etapa, base, antes, depois, obs=''):
    conferencia.append((etapa, base, antes, depois, obs))
    print(f'[{base}] {etapa}: {antes} -> {depois} linhas. {obs}')

Python 3.12.3 | pandas 3.0.6 | numpy 2.5.4


## Seção 3 — Leitura dos originais
Os arquivos originais em `data/raw/` não são editados; todo tratamento gera uma cópia.

In [2]:
rais_raw  = pd.read_csv(RAW/'rais_economistas_pb_2024.csv', encoding='utf-8-sig', dtype=str)
caged_raw = pd.read_csv(RAW/'caged_economistas_pb_2024.csv', sep=';', decimal=',', encoding='utf-8-sig',
                        dtype={'município':str,'cbo2002ocupação':str,'uf':str,'competênciamov':str})
dtb_raw   = pd.read_excel(RAW/'RELATORIO_DTB_BRASIL_MUNICIPIO.xls', header=6, dtype=str)
print('RAIS', rais_raw.shape, '| CAGED', caged_raw.shape, '| DTB', dtb_raw.shape)

RAIS (233, 62) | CAGED (41, 28) | DTB (5570, 9)


## Seção 4 — Diagnóstico inicial

In [3]:
def diagnostico(df, nome):
    print(f'=== {nome}: {df.shape[0]} linhas x {df.shape[1]} colunas | duplicatas exatas: {df.duplicated().sum()} | nulos totais: {int(df.isna().sum().sum())}')
    unicos = df.nunique()
    return pd.DataFrame({'dtype': df.dtypes.astype(str), 'nulos': df.isna().sum(), 'valores_distintos': unicos})

diag_rais = diagnostico(rais_raw, 'RAIS 2024')
diag_rais.to_csv(OUT/'diagnostico_rais.csv')
print('Colunas com um único valor (candidatas a remoção):', list(diag_rais.index[diag_rais.valores_distintos==1]))
diag_rais[diag_rais.nulos>0]

=== RAIS 2024: 233 linhas x 62 colunas | duplicatas exatas: 0 | nulos totais: 43
Colunas com um único valor (candidatas a remoção): ['Bairros SP - Código', 'Bairros Fortaleza - Código', 'Bairros RJ - Código', 'Distritos SP - Código', 'Ind CEI Vinculado - Código', 'Região Adm DF - Código', 'Tipo Estabelecimento - Código', 'Tipo Estabelecimento - Nome', 'Ano Chegada Brasil', 'Ind Trabalho Intermitente - Código', 'Ind Trabalho Parcial - Código', 'Ind Vínculo Abandonado - Código']


,dtype,nulos,valores_distintos
Vl Rem Dezembro Nom,str,43,169


In [4]:
diag_caged = diagnostico(caged_raw, 'CAGED 2024'); diag_caged.to_csv(OUT/'diagnostico_caged.csv'); diag_caged.head(10)

=== CAGED 2024: 41 linhas x 28 colunas | duplicatas exatas: 0 | nulos totais: 0


,dtype,nulos,valores_distintos
competênciamov,str,0,11
região,int64,0,1
uf,str,0,1
município,str,0,7
seção,str,0,9
subclasse,int64,0,27
saldomovimentação,int64,0,2
cbo2002ocupação,str,0,4
categoria,int64,0,1
graudeinstrução,int64,0,4


## Seção 5 — Tratamentos em etapas nomeadas
Uma função por etapa, cada uma imprimindo linhas antes e depois.

In [5]:
def padronizar_nomes(df):
    def snake(s):
        s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode().lower()
        return re.sub(r'_+','_', re.sub(r'[^a-z0-9]+','_', s)).strip('_')
    out = df.copy(); out.columns = [snake(c) for c in out.columns]; return out

def remover_colunas_constantes(df):
    const = [c for c in df.columns if df[c].nunique()==1]
    print('   removidas (valor único):', len(const))
    return df.drop(columns=const), const

def converter_tipos(df, texto, inteiros, decimais):
    out = df.copy()
    for c in texto:    out[c] = out[c].astype(str).str.strip()          # remove espaços (ex.: ' 84116')
    for c in inteiros: out[c] = pd.to_numeric(out[c], errors='coerce').astype('Int64')
    for c in decimais: out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def normalizar_municipio(df, col):
    out = df.copy(); out['cod_ibge6'] = out[col].astype(str).str.strip().str.zfill(6); return out

def filtrar_cbo_economista(df, col):
    out = df[df[col].astype(str).str.strip().isin(CBO_VALIDOS)].copy(); return out

def tratar_remuneracao(df, col_media, col_dez=None):
    out = df.copy()
    out['rem_media_informada'] = out[col_media] > 0                     # zero = não informado (sem imputar)
    out[col_media+'_original'] = out[col_media]
    out.loc[~out['rem_media_informada'], col_media] = np.nan
    pos = out[col_media].dropna(); q1,q3 = pos.quantile([.25,.75]); lim = q3 + 1.5*(q3-q1)
    out['rem_media_outlier_tukey'] = out[col_media] > lim
    print(f'   limite de Tukey: R$ {lim:,.2f} | outliers sinalizados (mantidos): {int(out.rem_media_outlier_tukey.sum())}')
    return out, lim

def tabela_municipios(dtb):
    pb = dtb[dtb['UF']=='25'][['Código Município Completo','Nome_Município']].copy()
    pb.columns = ['cod_ibge7','nome_municipio']; pb['cod_ibge6'] = pb.cod_ibge7.str[:6]
    return pb.reset_index(drop=True)

def juntar_municipios(df, pb, base):
    antes = len(df)
    assert pb.cod_ibge6.is_unique, 'chave da tabela auxiliar não é única'
    out = df.merge(pb[['cod_ibge6','nome_municipio']], on='cod_ibge6', how='left', validate='m:1')
    orfas = out.nome_municipio.isna().sum()
    registrar('junção com municípios (left join por cod_ibge6)', base, antes, len(out), f'{orfas} órfãs')
    return out, orfas

### 5.1 Municípios (IBGE)

In [6]:
municipios = tabela_municipios(dtb_raw)
registrar('filtro UF=25', 'IBGE', len(dtb_raw), len(municipios))
municipios.to_csv(PROC/'municipios_pb.csv', index=False); municipios.head()

[IBGE] filtro UF=25: 5570 -> 223 linhas. 


,cod_ibge7,nome_municipio,cod_ibge6
0,2500106,Água Branca,250010
1,2500205,Aguiar,250020
2,2500304,Alagoa Grande,250030
3,2500403,Alagoa Nova,250040
4,2500502,Alagoinha,250050


### 5.2 RAIS 2024

In [7]:
n0 = len(rais_raw)
rais = padronizar_nomes(rais_raw); registrar('padronizar_nomes', 'RAIS', n0, len(rais), f'{rais.shape[1]} colunas')
rais, const = remover_colunas_constantes(rais)
rais = converter_tipos(rais,
    texto=['cbo_2002_ocupacao_codigo','cnae_2_0_classe_codigo','cnae_2_0_subclasse_codigo','municipio_codigo','municipio_trab_codigo'],
    inteiros=['idade','qtd_hora_contr'],
    decimais=['vl_rem_media_nom','vl_rem_media_sm','vl_rem_dezembro_nom','vl_rem_dezembro_sm','tempo_emprego'])
rais = normalizar_municipio(rais, 'municipio_codigo')
n1 = len(rais); rais = filtrar_cbo_economista(rais, 'cbo_2002_ocupacao_codigo'); registrar('filtrar_cbo_economista', 'RAIS', n1, len(rais), 'família 2512')
rais, lim_rais = tratar_remuneracao(rais, 'vl_rem_media_nom')
rais, orfas_rais = juntar_municipios(rais, municipios, 'RAIS')
rais['ano_referencia'] = ANO
rais.shape

[RAIS] padronizar_nomes: 233 -> 233 linhas. 62 colunas
   removidas (valor único): 12
[RAIS] filtrar_cbo_economista: 233 -> 233 linhas. família 2512
   limite de Tukey: R$ 13,645.38 | outliers sinalizados (mantidos): 20
[RAIS] junção com municípios (left join por cod_ibge6): 233 -> 233 linhas. 0 órfãs


(233, 56)

### 5.3 CAGED 2024

In [8]:
n0 = len(caged_raw)
caged = caged_raw.rename(columns={'município':'municipio','cbo2002ocupação':'cbo','competênciamov':'competencia','saldomovimentação':'saldo','salário':'salario'}).copy()
caged = normalizar_municipio(caged, 'municipio')
n1 = len(caged); caged = filtrar_cbo_economista(caged, 'cbo'); registrar('filtrar_cbo_economista', 'CAGED', n1, len(caged), 'família 2512')
caged, lim_caged = tratar_remuneracao(caged, 'salario')
caged, orfas_caged = juntar_municipios(caged, municipios, 'CAGED')
caged['ano_referencia'] = ANO
caged.shape

[CAGED] filtrar_cbo_economista: 41 -> 41 linhas. família 2512
   limite de Tukey: R$ 6,990.57 | outliers sinalizados (mantidos): 2
[CAGED] junção com municípios (left join por cod_ibge6): 41 -> 41 linhas. 0 órfãs


(41, 34)

## Seção 6 — Testes de integridade
Cada teste é um `assert`: se falhar, a execução para.

In [9]:
testes = []
def teste(nome, cond, detalhe=''):
    assert cond, f'FALHOU: {nome} {detalhe}'
    testes.append((nome, 'aprovado', detalhe)); print('OK  ', nome, detalhe)

# Municípios (IBGE)
teste('IBGE: 223 municípios na PB', len(municipios)==223, f'({len(municipios)})')
teste('IBGE: códigos únicos (7 e 6 dígitos)', municipios.cod_ibge7.is_unique and municipios.cod_ibge6.is_unique)
teste('IBGE: todos começam com 25', municipios.cod_ibge7.str.startswith('25').all())

# RAIS
teste('RAIS: linhas depois do filtro = linhas lidas', len(rais)==len(rais_raw), f'({len(rais)})')
teste('RAIS: apenas CBO da família 2512', set(rais.cbo_2002_ocupacao_codigo) <= CBO_VALIDOS, str(sorted(set(rais.cbo_2002_ocupacao_codigo))))
teste('RAIS: sem duplicatas exatas', rais_raw.duplicated().sum()==0)
teste('RAIS: junção sem órfãs e sem multiplicar linhas', orfas_rais==0 and len(rais)==len(rais_raw))
teste('RAIS: todos os municípios da PB (começam com 25)', rais.cod_ibge6.str.startswith('25').all())
teste('RAIS: remuneração sem valores negativos', (rais.vl_rem_media_nom.dropna()>=0).all())
teste('RAIS: idade entre 14 e 100', rais.idade.between(14,100).all(), f'({rais.idade.min()} a {rais.idade.max()})')

# CAGED
teste('CAGED: apenas UF 25', caged_raw['uf'].eq('25').all())
teste('CAGED: apenas CBO da família 2512', set(caged.cbo) <= CBO_VALIDOS)
teste('CAGED: sem duplicatas exatas', caged_raw.duplicated().sum()==0)
teste('CAGED: junção sem órfãs e sem multiplicar linhas', orfas_caged==0 and len(caged)==len(caged_raw))
teste('CAGED: saldo = admissões - desligamentos', caged.saldo.sum() == (caged.saldo==1).sum() - (caged.saldo==-1).sum(), f'(saldo {caged.saldo.sum()})')
teste('CAGED: salário sem zero ou negativo', (caged_raw['salário']>0).all())

# Estoque vs. fluxo
teste('RAIS e CAGED mantidas em tabelas separadas (não somadas)', rais is not caged)
pd.DataFrame(testes, columns=['teste','resultado','detalhe']).to_csv(OUT/'testes_integridade.csv', index=False)
pd.DataFrame(conferencia, columns=['etapa','base','linhas_antes','linhas_depois','observacao']).to_csv(OUT/'conferencia_linhas.csv', index=False)

OK   IBGE: 223 municípios na PB (223)
OK   IBGE: códigos únicos (7 e 6 dígitos) 
OK   IBGE: todos começam com 25 
OK   RAIS: linhas depois do filtro = linhas lidas (233)
OK   RAIS: apenas CBO da família 2512 ['251205', '251210', '251215', '251220', '251225', '251235']
OK   RAIS: sem duplicatas exatas 
OK   RAIS: junção sem órfãs e sem multiplicar linhas 
OK   RAIS: todos os municípios da PB (começam com 25) 
OK   RAIS: remuneração sem valores negativos 
OK   RAIS: idade entre 14 e 100 (21 a 78)
OK   CAGED: apenas UF 25 
OK   CAGED: apenas CBO da família 2512 
OK   CAGED: sem duplicatas exatas 
OK   CAGED: junção sem órfãs e sem multiplicar linhas 
OK   CAGED: saldo = admissões - desligamentos (saldo 11)
OK   CAGED: salário sem zero ou negativo 
OK   RAIS e CAGED mantidas em tabelas separadas (não somadas) 


### Tabelas de conferência

In [10]:
print('RAIS por CBO:'); display(rais.cbo_2002_ocupacao_codigo.value_counts().rename('vinculos').to_frame())
print('RAIS por município:'); display(rais.nome_municipio.value_counts().rename('vinculos').to_frame())
print('Vínculo ativo em 31/12:'); display(rais.ind_vinculo_ativo_31_12_codigo.value_counts().rename('vinculos').to_frame())
print('Remuneração média nominal (R$, apenas informadas):'); display(rais.vl_rem_media_nom.describe().round(2).to_frame())
print('CAGED: movimentações por mês e saldo:'); display(caged.groupby('competencia').saldo.agg(linhas='size', saldo='sum'))

RAIS por CBO:


,vinculos
cbo_2002_ocupacao_codigo,
251205,87
251215,77
251225,51
251210,9
251220,8
251235,1


RAIS por município:


,vinculos
nome_municipio,
João Pessoa,159
Campina Grande,27
Cabedelo,11
Pedras de Fogo,9
Conde,6
Santa Rita,4
Caaporã,4
Rio Tinto,3
Santo André,2


Vínculo ativo em 31/12:


,vinculos
ind_vinculo_ativo_31_12_codigo,
1,200
0,33


Remuneração média nominal (R$, apenas informadas):


,vl_rem_media_nom
count,224.00
mean,5789.97
std,4556.33
min,1.10
25%,2795.88
50%,4127.00
75%,7135.68
max,24533.34


CAGED: movimentações por mês e saldo:


,linhas,saldo
competencia,,
202401,5,1
202402,2,-2
202403,1,1
202404,5,3
202405,1,1
202406,4,0
202407,4,0
202408,4,2
202409,6,2


## Seção 7 — Exportação e resumo das decisões

In [11]:
rais.to_csv(PROC/'rais_economistas_pb_2024_tratada.csv', index=False)
caged.to_csv(PROC/'caged_economistas_pb_2024_tratada.csv', index=False)
decisoes = pd.DataFrame([
 ('Nomes de colunas', 'padronizados em snake_case sem acento', 'colunas legíveis e sem espaços'),
 ('Colunas de valor único', f'{len(const)} removidas da RAIS', 'não carregam informação'),
 ('Códigos CNAE com espaço', 'espaços removidos, lidos como texto', 'evita falha em junções'),
 ('Remuneração zero', 'tratada como "não informado" (NaN), original preservada em *_original', 'sem imputação'),
 ('Remuneração extrema (Tukey)', 'sinalizada e mantida', 'sem exclusão sem justificativa'),
 ('Município', 'cod_ibge6 como chave de junção', 'RAIS/CAGED usam 6 dígitos; IBGE usa 7'),
 ('RAIS x CAGED', 'tabelas separadas', 'estoque e fluxo não se somam'),
], columns=['tema','decisao','motivo'])
decisoes.to_csv(OUT/'decisoes.csv', index=False)
print('Arquivos gravados em', PROC.resolve()); decisoes

Arquivos gravados em /home/claude/repo/data/processed


,tema,decisao,motivo
0,Nomes de colunas,padronizados em snake_case sem acento,colunas legíveis e sem espaços
1,Colunas de valor único,12 removidas da RAIS,não carregam informação
2,Códigos CNAE com espaço,"espaços removidos, lidos como texto",evita falha em junções
3,Remuneração zero,"tratada como ""não informado"" (NaN), original p...",sem imputação
4,Remuneração extrema (Tukey),sinalizada e mantida,sem exclusão sem justificativa
5,Município,cod_ibge6 como chave de junção,RAIS/CAGED usam 6 dígitos; IBGE usa 7
6,RAIS x CAGED,tabelas separadas,estoque e fluxo não se somam
